In [36]:
!pip install -q datasets transformers scikit-learn torch numpy matplotlib

In [1]:
"""
Scientific Citation-Support and Citation-Drift Verification
CPU-only; frozen transformer; chart generation.

Install:
    pip install datasets transformers scikit-learn torch numpy matplotlib

Run:
    python run_experiment_with_charts.py

Labels: 0=CONTRADICT, 1=NO_EVIDENCE, 2=SUPPORT
"""

import random
import re
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix, f1_score


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

SEED = 42
SAMPLE_SIZE = 100
JACCARD_THRESHOLD = 0.15
MODEL_NAME = "cross-encoder/nli-distilroberta-base"
MAX_LENGTH = 256
BATCH_SIZE = 16
CPU_THREADS = 4
DEVICE = torch.device("cpu")

LABEL_IDS = [0, 1, 2]
LABEL_NAMES = ["CONTRADICT", "NO_EVIDENCE", "SUPPORT"]

CHART_DIR = Path("experiment_charts")
CHART_DPI = 300

warnings.filterwarnings("ignore")


# ---------------------------------------------------------------------
# Data structures and common utilities
# ---------------------------------------------------------------------

@dataclass
class CitationSample:
    claim: str
    evidence: str
    label: int
    source: str


@dataclass
class SystemResult:
    name: str
    predictions: List[int]
    total_latency_ms: float
    sample_latencies_ms: Optional[List[float]] = None


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(CPU_THREADS)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def clean_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, (list, tuple)):
        value = " ".join(str(item) for item in value if item is not None)
    return re.sub(r"\s+", " ", str(value)).strip()


def ensure_list(value: Any) -> List[Any]:
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        return list(value)
    return [value]


def normalize_scifact_label(value: Any) -> Optional[int]:
    if value is None:
        return None
    label = str(value).strip().upper()
    return {
        "SUPPORT": 2, "SUPPORTS": 2, "SUPPORTED": 2,
        "CONTRADICT": 0, "CONTRADICTS": 0, "REFUTES": 0,
        "NOINFO": 1, "NO_INFO": 1, "NEUTRAL": 1,
        "NO_EVIDENCE": 1,
    }.get(label)


def normalize_scitail_label(
    raw_label: Any,
    dataset: Any,
    label_column: str,
) -> Optional[int]:
    # Integer values must be decoded using dataset metadata; never guess.
    if raw_label is None:
        return None
    if isinstance(raw_label, (int, np.integer)):
        try:
            feature = dataset.features.get(label_column)
            if feature is None or not hasattr(feature, "int2str"):
                return None
            raw_label = feature.int2str(int(raw_label))
        except (ValueError, KeyError, IndexError, TypeError):
            return None

    label = str(raw_label).strip().lower()
    return {
        "neutral": 1, "no_evidence": 1, "no evidence": 1,
        "entails": 2, "entailment": 2, "support": 2, "supports": 2,
    }.get(label)


# ---------------------------------------------------------------------
# SciFact corpus and claim extraction
# ---------------------------------------------------------------------

def load_scifact_corpus() -> Dict[str, str]:
    print("\n[DATA] Loading SciFact corpus...")
    corpus = load_dataset("allenai/scifact", "corpus", split="train")
    documents = {}

    for row in corpus:
        raw_id = row.get("doc_id")
        if raw_id is None:
            continue
        doc_id = str(raw_id).strip()
        title = clean_text(row.get("title", ""))
        abstract = clean_text(row.get("abstract", []))
        text = clean_text(f"{title}. {abstract}")
        if doc_id and text:
            documents[doc_id] = text

    print(f"[DATA] Corpus documents: {len(documents)}")
    if not documents:
        raise RuntimeError("SciFact corpus contained no usable documents.")
    return documents


def add_nested_evidence(evidence: Any, group: dict) -> None:
    """Read original nested evidence: {doc_id: [{label: ..., sentences: ...}]}."""
    if isinstance(evidence, dict):
        iterator = evidence.items()
    elif isinstance(evidence, list):
        iterator = []
        for item in evidence:
            if isinstance(item, dict):
                doc_id = item.get("doc_id", item.get("evidence_doc_id"))
                if doc_id is not None:
                    iterator.append((doc_id, [item]))
    else:
        return

    for raw_id, annotations in iterator:
        doc_id = str(raw_id).strip()
        if not doc_id:
            continue
        group["document_ids"].add(doc_id)
        for annotation in ensure_list(annotations):
            if isinstance(annotation, dict):
                label = normalize_scifact_label(annotation.get("label"))
                if label is not None:
                    group["annotations"][doc_id].append(label)


def add_flattened_evidence(row: dict, group: dict) -> None:
    """Read scalar/list evidence_doc_id and evidence_label fields."""
    doc_ids, labels = [], []

    for field in ("evidence_doc_id", "evidence_doc_ids"):
        doc_ids.extend(ensure_list(row.get(field)))
    for field in ("evidence_label", "evidence_labels"):
        labels.extend(ensure_list(row.get(field)))

    for raw_id in doc_ids:
        if raw_id is not None and str(raw_id).strip():
            group["document_ids"].add(str(raw_id).strip())

    if doc_ids and labels and len(doc_ids) != len(labels):
        group["warnings"]["unaligned_flattened_fields"] += 1

    # Positional alignment is assumed when these fields are list-valued.
    for raw_id, raw_label in zip(doc_ids, labels):
        if raw_id is None:
            continue
        doc_id = str(raw_id).strip()
        label = normalize_scifact_label(raw_label)
        if doc_id and label is not None:
            group["annotations"][doc_id].append(label)


def extract_scifact_samples(
    claims_dataset: Any,
    document_lookup: Dict[str, str],
) -> List[CitationSample]:
    grouped = {}
    diagnostics = Counter()

    for row in claims_dataset:
        claim = clean_text(row.get("claim", ""))
        if not claim:
            diagnostics["empty_claim"] += 1
            continue

        claim_id = row.get("id", row.get("claim_id"))
        key = str(claim_id if claim_id is not None else claim)

        if key not in grouped:
            grouped[key] = {
                "claim": claim,
                "document_ids": set(),
                "annotations": defaultdict(list),
                "warnings": Counter(),
            }

        group = grouped[key]
        for raw_id in ensure_list(row.get("cited_doc_ids")):
            if raw_id is not None and str(raw_id).strip():
                group["document_ids"].add(str(raw_id).strip())

        add_nested_evidence(row.get("evidence"), group)
        add_flattened_evidence(row, group)

    samples = []
    seen_pairs = set()

    for group in grouped.values():
        claim = group["claim"]

        for doc_id in sorted(group["document_ids"]):
            if doc_id not in document_lookup:
                diagnostics["missing_document"] += 1
                continue

            evidence = clean_text(document_lookup[doc_id])
            if not evidence:
                diagnostics["empty_evidence"] += 1
                continue

            pair_key = (claim, doc_id)
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)

            labels = group["annotations"].get(doc_id, [])
            if labels:
                counts = Counter(labels)
                max_count = max(counts.values())
                winners = [label for label, count in counts.items()
                           if count == max_count]
                if len(winners) != 1:
                    diagnostics["ambiguous_annotation"] += 1
                    continue
                gold_label = winners[0]
            else:
                # Operational definition: cited but not evidence-annotated.
                gold_label = 1

            samples.append(CitationSample(
                claim=claim,
                evidence=evidence,
                label=gold_label,
                source="SciFact",
            ))

    print("\n[DATA] SciFact extraction diagnostics")
    print(f"Grouped claims: {len(grouped)}")
    print(f"Extracted pairs: {len(samples)}")
    print(f"Skipped/diagnostic counts: {dict(diagnostics)}")
    return samples


def load_scifact(sample_size: int) -> List[CitationSample]:
    document_lookup = load_scifact_corpus()

    for split in ("validation", "train"):
        try:
            print(f"\n[DATA] Loading SciFact claims split: {split}")
            claims = load_dataset("allenai/scifact", "claims", split=split)
            print(f"[DATA] Claim columns: {claims.column_names}")
            print(f"[DATA] Claim features: {claims.features}")

            samples = extract_scifact_samples(claims, document_lookup)
            if samples:
                rng = np.random.default_rng(SEED)
                indices = rng.permutation(len(samples))[:sample_size]
                selected = [samples[int(i)] for i in indices]
                print(f"[DATA] Selected {len(selected)} SciFact pairs.")
                return selected

            print(f"[WARNING] No valid pairs from {split}.")
        except Exception as exc:
            print(f"[WARNING] SciFact {split} loading failed: {exc}")

    raise RuntimeError("SciFact could not produce valid claim-document pairs.")


# ---------------------------------------------------------------------
# SciTail fallback
# ---------------------------------------------------------------------

def load_scitail(sample_size: int) -> List[CitationSample]:
    print("\n[DATA] Loading SciTail fallback...")
    dataset = load_dataset("allenai/scitail", "snli_format", split="test")

    print(f"[DATA] Rows: {len(dataset)}")
    print(f"[DATA] Columns: {dataset.column_names}")
    print(f"[DATA] Features: {dataset.features}")

    if len(dataset) == 0:
        raise RuntimeError("SciTail test split is empty.")

    columns = set(dataset.column_names)
    candidates = [("premise", "hypothesis"), ("sentence1", "sentence2")]
    pair = next(((p, h) for p, h in candidates if p in columns and h in columns), None)
    if pair is None:
        raise RuntimeError(f"Unsupported SciTail columns: {dataset.column_names}")

    evidence_column, claim_column = pair
    label_candidates = ("label", "gold_label", "gold_label_text", "entailment_judgment")
    label_column = next((name for name in label_candidates if name in columns), None)
    if label_column is None:
        raise RuntimeError("No recognized SciTail label column.")

    print(f"[DATA] Evidence column: {evidence_column}")
    print(f"[DATA] Claim column: {claim_column}")
    print(f"[DATA] Label column: {label_column}")
    print(f"[DEBUG] First row: {dataset[0]}")

    samples = []
    rejected = Counter()
    observed_labels = Counter()

    for row in dataset:
        evidence = clean_text(row.get(evidence_column))
        claim = clean_text(row.get(claim_column))
        if not evidence or not claim:
            rejected["empty_text"] += 1
            continue

        raw_label = row.get(label_column)
        observed_labels[str(raw_label)] += 1
        label = normalize_scitail_label(raw_label, dataset, label_column)
        if label is None:
            rejected["unrecognized_label"] += 1
            continue

        samples.append(CitationSample(
            claim=claim,
            evidence=evidence,
            label=label,
            source="SciTail",
        ))

    print(f"[DATA] Observed labels: {dict(observed_labels)}")
    print(f"[DATA] Rejected rows: {dict(rejected)}")
    print(f"[DATA] Valid rows: {len(samples)}")

    if not samples:
        raise RuntimeError("SciTail extraction produced zero valid samples.")

    rng = np.random.default_rng(SEED)
    indices = rng.permutation(len(samples))[:sample_size]
    return [samples[int(i)] for i in indices]


def load_experiment_data(sample_size: int) -> List[CitationSample]:
    try:
        samples = load_scifact(sample_size)
        if samples:
            return samples
    except Exception as exc:
        print(f"\n[WARNING] SciFact unavailable: {exc}")

    print("\n[INFO] Activating SciTail fallback.")
    samples = load_scitail(sample_size)
    if not samples:
        raise RuntimeError("Both datasets returned zero usable samples.")
    return samples


# ---------------------------------------------------------------------
# Dataset report
# ---------------------------------------------------------------------

def print_dataset_statistics(samples: List[CitationSample]) -> None:
    distribution = Counter(sample.label for sample in samples)

    print("\n" + "=" * 72)
    print("DATASET SANITY CHECK")
    print("=" * 72)
    print(f"Dataset source: {samples[0].source}")
    print(f"Total samples: {len(samples)}")

    for label_id, label_name in zip(LABEL_IDS, LABEL_NAMES):
        count = distribution.get(label_id, 0)
        pct = 100.0 * count / len(samples)
        print(f"{label_name:<18} count={count:>4}  percentage={pct:>7.2f}%")
    print("=" * 72)


# ---------------------------------------------------------------------
# System 0: Jaccard lexical baseline
# ---------------------------------------------------------------------

def tokenize(text: str) -> set:
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def jaccard_similarity(claim: str, evidence: str) -> float:
    claim_tokens = tokenize(claim)
    evidence_tokens = tokenize(evidence)
    union = claim_tokens | evidence_tokens
    if not union:
        return 0.0
    return len(claim_tokens & evidence_tokens) / len(union)


def run_jaccard(samples: List[CitationSample]) -> SystemResult:
    predictions, sample_times = [], []
    total_start = time.perf_counter()

    for sample in samples:
        start = time.perf_counter()
        similarity = jaccard_similarity(sample.claim, sample.evidence)
        predictions.append(2 if similarity >= JACCARD_THRESHOLD else 1)
        sample_times.append((time.perf_counter() - start) * 1000.0)

    total_ms = (time.perf_counter() - total_start) * 1000.0
    return SystemResult("Jaccard", predictions, total_ms, sample_times)


# ---------------------------------------------------------------------
# System 1: frozen Cross-Encoder
# ---------------------------------------------------------------------

def normalize_model_label(label: Any) -> Optional[int]:
    text = str(label).strip().lower()
    if "contradict" in text or "refut" in text:
        return 0
    if "entail" in text or "support" in text:
        return 2
    if "neutral" in text or "no_evidence" in text or "no evidence" in text:
        return 1
    return None


def get_model_label_mapping(model) -> Dict[int, int]:
    id2label = model.config.id2label or {}
    mapping = {}

    for index in range(model.config.num_labels):
        raw = id2label.get(index, id2label.get(str(index), ""))
        normalized = normalize_model_label(raw)
        if normalized is not None:
            mapping[index] = normalized

    if len(mapping) != model.config.num_labels:
        if MODEL_NAME == "cross-encoder/nli-distilroberta-base" and model.config.num_labels == 3:
            print("[WARNING] Generic model labels; using documented checkpoint order.")
            mapping = {0: 0, 1: 2, 2: 1}
        else:
            raise RuntimeError(f"Cannot safely resolve model labels: {id2label}")

    if set(mapping.values()) != {0, 1, 2}:
        raise RuntimeError(f"Unexpected NLI label mapping: {mapping}")

    print("[MODEL] Logit mapping:")
    for index, label_id in mapping.items():
        print(f"  logit {index} -> {LABEL_NAMES[label_id]}")
    return mapping


def load_nli_model():
    print(f"\n[MODEL] Loading {MODEL_NAME} on {DEVICE}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
        model.to(DEVICE)
        model.eval()
        for parameter in model.parameters():
            parameter.requires_grad = False
        return tokenizer, model
    except Exception as exc:
        raise RuntimeError(f"Could not load pretrained NLI model: {exc}") from exc


def run_cross_encoder(
    samples: List[CitationSample],
    tokenizer,
    model,
) -> SystemResult:
    mapping = get_model_label_mapping(model)
    predictions = []
    premises = [sample.evidence for sample in samples]
    hypotheses = [sample.claim for sample in samples]

    total_start = time.perf_counter()

    with torch.no_grad():
        for start in range(0, len(samples), BATCH_SIZE):
            stop = min(start + BATCH_SIZE, len(samples))
            encoded = tokenizer(
                premises[start:stop],
                hypotheses[start:stop],
                truncation=True,
                max_length=MAX_LENGTH,
                padding=True,
                return_tensors="pt",
            )
            encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
            logits = model(**encoded).logits
            raw_predictions = torch.argmax(logits, dim=1).cpu().tolist()
            predictions.extend(mapping[int(index)] for index in raw_predictions)

    total_ms = (time.perf_counter() - total_start) * 1000.0
    return SystemResult("Cross-Encoder", predictions, total_ms)


# ---------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------

def classification_metrics(y_true: List[int], y_pred: List[int]) -> dict:
    return classification_report(
        y_true,
        y_pred,
        labels=LABEL_IDS,
        target_names=LABEL_NAMES,
        output_dict=True,
        zero_division=0,
    )


def evaluate_system(result: SystemResult, y_true: List[int]) -> float:
    print("\n" + "=" * 76)
    print(f"EVALUATION: {result.name}")
    print("=" * 76)

    print(classification_report(
        y_true,
        result.predictions,
        labels=LABEL_IDS,
        target_names=LABEL_NAMES,
        zero_division=0,
        digits=4,
    ))

    macro_f1 = f1_score(
        y_true,
        result.predictions,
        labels=LABEL_IDS,
        average="macro",
        zero_division=0,
    )
    print(f"Macro-Averaged F1: {macro_f1:.4f}")

    matrix = confusion_matrix(y_true, result.predictions, labels=LABEL_IDS)
    print("\nConfusion matrix (rows=ground truth, columns=predicted):")
    print(f"{'':>18}" + "".join(f"{name:>18}" for name in LABEL_NAMES))
    for name, row in zip(LABEL_NAMES, matrix):
        print(f"{name:>18}" + "".join(f"{int(v):>18}" for v in row))

    n = len(y_true)
    print(f"\nTotal measured latency: {result.total_latency_ms:.3f} ms")
    print(f"Amortized latency/sample: {result.total_latency_ms / n:.3f} ms")
    if result.sample_latencies_ms is not None:
        print(f"Mean per-sample latency: {np.mean(result.sample_latencies_ms):.6f} ms")
        print(f"Median per-sample latency: {np.median(result.sample_latencies_ms):.6f} ms")
    return float(macro_f1)


# ---------------------------------------------------------------------
# Chart generation
# ---------------------------------------------------------------------

def save_figure(fig, filename: str) -> None:
    CHART_DIR.mkdir(parents=True, exist_ok=True)
    output = CHART_DIR / filename
    fig.savefig(output, dpi=CHART_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"[CHART] Saved: {output.resolve()}")


def plot_class_distribution(samples: List[CitationSample]) -> None:
    counts = Counter(sample.label for sample in samples)
    values = [counts.get(label_id, 0) for label_id in LABEL_IDS]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(LABEL_NAMES, values)
    ax.set_title("Ground-Truth Class Distribution")
    ax.set_xlabel("Ground-truth class")
    ax.set_ylabel("Number of samples")
    ax.grid(axis="y", alpha=0.25)

    for bar, value in zip(bars, values):
        ax.annotate(
            str(value),
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            ha="center",
            va="bottom",
            xytext=(0, 4),
            textcoords="offset points",
        )
    save_figure(fig, "01_dataset_class_distribution.png")


def plot_macro_f1(results: List[SystemResult], y_true: List[int]) -> None:
    names = [result.name for result in results]
    scores = [
        f1_score(y_true, result.predictions, labels=LABEL_IDS,
                 average="macro", zero_division=0)
        for result in results
    ]

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(names, scores)
    ax.set_title("Macro-F1 Comparison")
    ax.set_ylabel("Macro-F1")
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.25)

    for bar, score in zip(bars, scores):
        ax.annotate(
            f"{score:.3f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            ha="center",
            va="bottom",
            xytext=(0, 4),
            textcoords="offset points",
        )
    save_figure(fig, "02_macro_f1_comparison.png")


def plot_per_class_metrics(results: List[SystemResult], y_true: List[int]) -> None:
    metric_names = ["precision", "recall", "f1-score"]
    x = np.arange(len(LABEL_NAMES))
    width = 0.12
    total_bars = len(results) * len(metric_names)
    offsets = (np.arange(total_bars) - (total_bars - 1) / 2) * width

    fig, ax = plt.subplots(figsize=(12, 6))
    bar_index = 0

    for result in results:
        report = classification_metrics(y_true, result.predictions)
        for metric in metric_names:
            values = [report[label][metric] for label in LABEL_NAMES]
            ax.bar(
                x + offsets[bar_index],
                values,
                width,
                label=f"{result.name} - {metric.capitalize()}",
            )
            bar_index += 1

    ax.set_title("Per-Class Precision, Recall and F1")
    ax.set_xlabel("Ground-truth class")
    ax.set_ylabel("Score")
    ax.set_xticks(x)
    ax.set_xticklabels(LABEL_NAMES)
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(ncol=2, fontsize=8)
    save_figure(fig, "03_per_class_metrics.png")


def plot_confusion_matrices(
    results: List[SystemResult],
    y_true: List[int],
    normalize: bool = False,
) -> None:
    fig, axes = plt.subplots(
        1, len(results),
        figsize=(6 * len(results), 5),
        squeeze=False,
    )

    for ax, result in zip(axes[0], results):
        matrix = confusion_matrix(
            y_true, result.predictions, labels=LABEL_IDS
        ).astype(float)

        if normalize:
            row_sums = matrix.sum(axis=1, keepdims=True)
            matrix = np.divide(
                matrix,
                row_sums,
                out=np.zeros_like(matrix),
                where=row_sums != 0,
            )

        image = ax.imshow(matrix, interpolation="nearest", aspect="auto")
        ax.set_title(result.name)
        ax.set_xlabel("Predicted label")
        ax.set_ylabel("Ground-truth label")
        ax.set_xticks(range(3))
        ax.set_yticks(range(3))
        ax.set_xticklabels(LABEL_NAMES, rotation=25, ha="right")
        ax.set_yticklabels(LABEL_NAMES)

        threshold = matrix.max() / 2 if matrix.size else 0
        for row in range(3):
            for col in range(3):
                cell = f"{matrix[row, col]:.2f}" if normalize else f"{int(matrix[row, col])}"
                ax.text(
                    col, row, cell,
                    ha="center", va="center",
                    color="white" if matrix[row, col] > threshold else "black",
                )

        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

    if normalize:
        fig.suptitle("Row-Normalized Confusion Matrices")
        filename = "05_confusion_matrices_row_normalized.png"
    else:
        fig.suptitle("Confusion Matrices (Raw Counts)")
        filename = "04_confusion_matrices_counts.png"

    fig.tight_layout()
    save_figure(fig, filename)


def plot_latency(results: List[SystemResult], sample_count: int) -> None:
    names = [result.name for result in results]
    per_sample = [result.total_latency_ms / sample_count for result in results]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(names, per_sample)
    ax.set_title("Amortized Inference Latency")
    ax.set_xlabel("System")
    ax.set_ylabel("Milliseconds per sample")
    ax.grid(axis="y", alpha=0.25)

    for bar, value in zip(bars, per_sample):
        ax.annotate(
            f"{value:.3f} ms",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            ha="center",
            va="bottom",
            xytext=(0, 4),
            textcoords="offset points",
        )
    save_figure(fig, "06_latency_comparison.png")


def generate_all_charts(
    samples: List[CitationSample],
    results: List[SystemResult],
    y_true: List[int],
) -> None:
    CHART_DIR.mkdir(parents=True, exist_ok=True)
    print("\n[CHART] Generating charts...")

    plot_class_distribution(samples)
    plot_macro_f1(results, y_true)
    plot_per_class_metrics(results, y_true)
    plot_confusion_matrices(results, y_true, normalize=False)
    plot_confusion_matrices(results, y_true, normalize=True)
    plot_latency(results, len(samples))

    print(f"[CHART] All charts saved to: {CHART_DIR.resolve()}")


# ---------------------------------------------------------------------
# Comparative summary and main
# ---------------------------------------------------------------------

def print_comparative_summary(results: List[SystemResult], y_true: List[int]) -> None:
    print("\n" + "=" * 76)
    print("FINAL COMPARATIVE RESULTS")
    print("=" * 76)
    print(f"{'Metric':<28}" + "".join(f"{result.name:>22}" for result in results))
    print("-" * 76)

    metric_rows = [
        ("Macro F1", [
            f1_score(y_true, r.predictions, labels=LABEL_IDS,
                     average="macro", zero_division=0)
            for r in results
        ]),
        ("Total latency (ms)", [r.total_latency_ms for r in results]),
        ("Latency/sample (ms)", [
            r.total_latency_ms / len(y_true) for r in results
        ]),
    ]

    for metric, values in metric_rows:
        print(f"{metric:<28}" + "".join(f"{value:>22.4f}" for value in values))
    print("=" * 76)


def main() -> None:
    set_seed()

    print("\n" + "#" * 76)
    print("SCIENTIFIC CITATION-SUPPORT AND CITATION-DRIFT EXPERIMENT")
    print("#" * 76)
    print(f"Seed: {SEED}")
    print(f"Requested sample size: {SAMPLE_SIZE}")
    print(f"Device: {DEVICE}")
    print(f"Jaccard threshold: {JACCARD_THRESHOLD}")
    print(f"Transformer: {MODEL_NAME}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Maximum sequence length: {MAX_LENGTH}")

    # 1. Dataset ingestion.
    samples = load_experiment_data(SAMPLE_SIZE)
    if not samples:
        raise RuntimeError("No samples available for evaluation.")

    # 2. Dataset sanity report.
    print_dataset_statistics(samples)
    y_true = [sample.label for sample in samples]

    # 3. Jaccard baseline.
    print("\n[EXPERIMENT] Running Jaccard baseline...")
    jaccard_result = run_jaccard(samples)
    evaluate_system(jaccard_result, y_true)
    results = [jaccard_result]

    # 4. Frozen transformer. If download/inference fails, retain baseline output.
    try:
        tokenizer, model = load_nli_model()

        warmup = tokenizer(
            ["A scientific statement."],
            ["A scientific statement."],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        )
        with torch.no_grad():
            _ = model(**warmup)

        print("[MODEL] Warm-up complete.")
        nli_result = run_cross_encoder(samples, tokenizer, model)
        evaluate_system(nli_result, y_true)
        results.append(nli_result)

    except Exception as exc:
        print("\n[WARNING] Transformer evaluation failed.")
        print(f"Reason: {exc}")
        print("Charts will be generated for systems that completed.")

    # 5. Comparative report and charts.
    print_comparative_summary(results, y_true)
    generate_all_charts(samples, results, y_true)

    print("\n[EXPERIMENT] Completed.")
    print(f"[OUTPUT] Charts directory: {CHART_DIR.resolve()}")


if __name__ == "__main__":
    main()



############################################################################
SCIENTIFIC CITATION-SUPPORT AND CITATION-DRIFT EXPERIMENT
############################################################################
Seed: 42
Requested sample size: 100
Device: cpu
Jaccard threshold: 0.15
Transformer: cross-encoder/nli-distilroberta-base
Batch size: 16
Maximum sequence length: 256

[DATA] Loading SciFact corpus...


README.md:   0%|          | 0.00/8.06k [00:00<?, ?B/s]

scifact.py:   0%|          | 0.00/7.92k [00:00<?, ?B/s]


[WARNING] SciFact unavailable: Dataset scripts are no longer supported, but found scifact.py

[INFO] Activating SciTail fallback.

[DATA] Loading SciTail fallback...


README.md:   0%|          | 0.00/10.1k [00:00<?, ?B/s]

snli_format/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.42MB            

... (15 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  328MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[MODEL] Warm-up complete.
[MODEL] Logit mapping:
  logit 0 -> CONTRADICT
  logit 1 -> SUPPORT
  logit 2 -> NO_EVIDENCE

EVALUATION: Cross-Encoder
              precision    recall  f1-score   support

  CONTRADICT     0.0000    0.0000    0.0000         0
 NO_EVIDENCE     0.8000    0.5902    0.6792        61
     SUPPORT     0.7368    0.7179    0.7273        39

    accuracy                         0.6400       100
   macro avg     0.5123    0.4360    0.4688       100
weighted avg     0.7754    0.6400    0.6980       100

Macro-Averaged F1: 0.4688

Confusion matrix (rows=ground truth, columns=predicted):
                          CONTRADICT       NO_EVIDENCE           SUPPORT
        CONTRADICT                 0                 0                 0
       NO_EVIDENCE                15                36                10
           SUPPORT                 2                 9                28

Total measured latency: 61810.569 ms
Amortized latency/sample: 618.106 ms

FINAL COMPARATIVE RES